<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Qwen_rank_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx
# If the import in the next cells fails with a Qwen2.5-Omni error, use the dev build instead:
# !pip install -q git+https://github.com/huggingface/transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 119.0 MB/s eta 0:00:00


In [2]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


In [3]:
import torch
from transformers import (Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor,
                          BitsAndBytesConfig)

assert torch.cuda.is_available(), "No GPU: set the runtime type to T4 GPU and restart."

model_id = "Qwen/Qwen2.5-Omni-3B"
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

def load(quantize_4bit=False):
    kwargs = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            llm_int8_skip_modules=["audio_tower", "visual", "lm_head"])
    return Qwen2_5OmniForConditionalGeneration.from_pretrained(model_id, **kwargs)

try:
    model = load(quantize_4bit=False)
    print("Loaded fp16")
except torch.OutOfMemoryError:
    import gc; gc.collect(); torch.cuda.empty_cache()
    model = load(quantize_4bit=True)
    print("fp16 didn't fit, loaded 4-bit")

# We only generate text, so drop the speech output modules to free VRAM
if hasattr(model, "disable_talker"):
    model.disable_talker()
torch.cuda.empty_cache()

print("device:", next(model.parameters()).device)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

chat_template.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json:   0%|          | 0.00/6.47k [00:00<?, ?B/s]

...

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt: reconstructing file:   0%|          |  0.00B /  260kB            

spk_dict.pt: downloading bytes:           |  0.00B            

Loaded fp16
device: cuda:0
GPU memory used: 9.4 GB


In [4]:
import os, gc, glob, subprocess, fitz, docx
from qwen_omni_utils import process_mm_info

IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
TXT = {".txt", ".md", ".csv", ".json", ".html"}

MAX_CHARS = 12000     # per text piece
MAX_PIECES = 8        # long documents are summarized piece by piece, capped at this many

# Text-only output, so no need for the default "speech" persona. Short answers = faster.
SYSTEM = {"role": "system", "content": [{"type": "text",
  "text": "You are a concise assistant. Answer directly in plain text. Never ask the user questions."}]}
PROMPT = "What is this about? Give a short summary of the main topic and key points in at most 4 sentences."

# Video quality tiers, tried in order if the GPU runs out of memory
VIDEO_SETTINGS = [
    {"nframes": 24, "max_pixels": 360 * 420},
    {"nframes": 12, "max_pixels": 224 * 224},
    {"nframes": 8,  "max_pixels": 168 * 168},
]

def _text_content(text):
    # Long text: summarize each piece first, then summarize the summaries
    if len(text) > MAX_CHARS:
        pieces = [text[i:i + MAX_CHARS] for i in range(0, len(text), MAX_CHARS)][:MAX_PIECES]
        notes = []
        for i, p in enumerate(pieces):
            note = run_content([{"type": "text",
                                 "text": f"{p}\n\nSummarize this part in 3 sentences."}], 120)
            notes.append(f"Part {i + 1}: {note}")
        text = "\n".join(notes)
    return [{"type": "text", "text": f"{text}\n\n{PROMPT}"}]

def _pdf_content(path):
    with fitz.open(path) as doc:
        text = "\n".join(page.get_text() for page in doc)
        if len(text.strip()) > 200:
            return _text_content(text)
        # scanned PDF: send the first 3 pages as images
        content = []
        for i, page in enumerate(doc):
            if i >= 3: break
            png = f"/content/_page{i}.png"
            page.get_pixmap(dpi=100).save(png)
            content.append({"type": "image", "image": png})
    return content + [{"type": "text", "text": PROMPT}]

def build_content(src):
    if not src.startswith("http") and not os.path.exists(src):
        return _text_content(src)          # plain text typed in directly
    ext = os.path.splitext(src.split("?")[0])[1].lower()
    if ext in IMG:
        return [{"type": "image", "image": src}, {"type": "text", "text": PROMPT}]
    if ext in VID:
        return [{"type": "video", "video": src, **VIDEO_SETTINGS[0]},
                {"type": "text", "text": PROMPT}]
    if ext in AUD:
        return [{"type": "audio", "audio": src}, {"type": "text", "text": PROMPT}]
    if ext == ".pdf":
        return _pdf_content(src)
    if ext == ".docx":
        return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in TXT:
        with open(src, errors="ignore") as f:
            return _text_content(f.read())
    raise ValueError(f"Unsupported file type: {ext}")

In [5]:
def _run(conversation, max_new_tokens):
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to("cuda").to(torch.float16)   # explicit device, never model.device
    print("input tokens:", inputs.input_ids.shape[1])
    with torch.inference_mode():
        out = model.thinker.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                  skip_special_tokens=True)[0].strip()

def run_content(content, max_new_tokens=200):
    is_video = any(c["type"] == "video" for c in content)
    for tier in (VIDEO_SETTINGS if is_video else [None]):
        if tier:
            for c in content:
                if c["type"] == "video":
                    c.update(tier)
        try:
            return _run([SYSTEM, {"role": "user", "content": content}], max_new_tokens)
        except torch.OutOfMemoryError:
            print("Out of memory, retrying with smaller settings...")
        finally:
            gc.collect(); torch.cuda.empty_cache()
    return "Still out of memory."

def what_is_this(src, max_new_tokens=200):
    return run_content(build_content(src), max_new_tokens)

In [6]:
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def split_video(path, chunk_sec=20, out_dir="/content/chunks"):
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"):
        os.remove(f)
    # "-c copy" = no re-encoding, so splitting takes seconds. Cuts land on keyframes,
    # so chunk lengths are approximate. If a chunk fails to decode, swap in
    # ["-c:v", "libx264", "-preset", "veryfast"] for ["-c", "copy"].
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an",
                    "-c", "copy", "-f", "segment", "-segment_time", str(chunk_sec),
                    "-reset_timestamps", "1", f"{out_dir}/chunk_%03d.mp4"], check=True)
    return sorted(glob.glob(f"{out_dir}/*.mp4"))

def what_is_this_video(src, chunk_sec=20):
    if video_duration(src) <= chunk_sec * 1.5:
        return what_is_this(src)
    notes = []
    for i, chunk in enumerate(split_video(src, chunk_sec)):
        content = [{"type": "video", "video": chunk},
                   {"type": "text", "text": "Describe what happens in this clip in 2 sentences."}]
        notes.append(f"Part {i + 1}: {run_content(content, 80)}")
        print(notes[-1])
    summary_prompt = ("These are notes on consecutive parts of one video:\n\n"
                      + "\n".join(notes) + "\n\n" + PROMPT)
    return run_content([{"type": "text", "text": summary_prompt}], 200)

In [7]:
print(what_is_this("Photosynthesis is how plants turn sunlight, water and CO2 into sugar and oxygen."))

input tokens: 71
Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose (sugar) and oxygen. It is a fundamental biological process that sustains life on Earth. Photosynthesis occurs primarily in the chloroplasts of plant cells, using chlorophyll to capture light energy. The main products are glucose, which serves as energy for the plant, and oxygen, which is released into the atmosphere.


In [8]:
print(what_is_this("/content/7_test1.png"))

input tokens: 186
This image is a whimsical, surreal illustration featuring a fantastical landscape with a variety of elements. At the center, there's a stack of pancakes topped with strawberries and a smiling face, suggesting a playful and imaginative setting. The background includes a rainbow, mushrooms, and a flying cat with wings, adding to the dreamlike atmosphere. The scene is vibrant and colorful, blending elements of nature, fantasy, and whimsy, creating a visually engaging and imaginative world.


In [9]:
print(what_is_this("/content/9_test3.jpeg"))

input tokens: 440
The image depicts a woman walking through a crowd of people outdoors. She is wearing a green top and has long hair. The background shows trees and a blurred view of the surroundings, suggesting a public or urban setting. The focus is on the woman, with the crowd slightly out of focus, emphasizing her presence.


In [10]:
print(what_is_this_video("/content/dog.mp4"))

qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


input tokens: 2216
The image captures a dog, likely a Border Collie, energetically running along the edge of a body of water, possibly a lake or river. The dog's fur is wet, suggesting it has been playing in the water. The dog's tail is raised, and it appears to be looking towards the water, possibly tracking something or simply enjoying the moment. The background is bright and slightly overexposed, indicating that the photo was taken during the daytime with sunlight reflecting off the water. The dog's harness is visible, indicating it might be a pet or part of a training session.


In [11]:
print(what_is_this_video("/content/movie_clip1.mp4"))

input tokens: 2204
Part 1: A woman in a school uniform is introduced to two women.
input tokens: 2204
Part 2: A woman in a black jacket and scarf is talking to a woman in a brown dress. The woman in the brown dress is holding a phone.
input tokens: 2204
Part 3: Three women are in a room. One of them is wearing a school uniform.
input tokens: 2204
Part 4: A girl in a kilt runs into a room and is greeted by her mother.
input tokens: 2204
Part 5: A woman and a girl sit in chairs in a room. The woman speaks and the girl leans back in her chair.
input tokens: 182
The video appears to depict a series of scenes involving women in various settings, including a school, a room, and a home. It includes interactions between women in school uniforms, a woman in a black jacket, and a woman in a brown dress. Additionally, there is a scene with a girl in a kilt running into a room and being greeted by her mother. The video also shows a woman and a girl sitting in chairs, with the woman speaking and th

In [12]:
print(what_is_this_video("/content/movie_hindi.mp4"))

input tokens: 2216
The video showcases a series of scenes featuring a woman in various settings and outfits. It begins with her shopping in a store, then transitions to a snowy outdoor scene where she is seen with a group of people. The video continues with a scene in a room where she is sitting with a man, followed by a festive outdoor celebration where she is surrounded by people covered in yellow powder. The final scene shows her in a traditional setting, smiling and interacting with others.
